# Micro-Expression Recognition — Baseline Model
## 6-Channel Swin Transformer (Onset + Apex Frame Concatenation)

---

### What this notebook does

This is the **baseline model** for micro-expression recognition (MER) on the CASME3 dataset.  
It is designed to be a solid starting point — strong enough to be a meaningful benchmark, simple enough to understand every step.

---

### Why 6-channel input?

A standard image is 3 channels (RGB). We feed the model **two frames stacked together = 6 channels**:

| Channels | Content | Purpose |
|---|---|---|
| 1–3 | **Onset frame** (neutral face) | Gives the model a reference — what this person looks like at rest |
| 4–6 | **Apex frame** (peak expression) | The most information-rich moment of the micro-expression |

By seeing both frames simultaneously, the model can learn *differences* between neutral and expressive states, rather than trying to classify the apex frame in isolation.  
This is a middle ground between:
- **Apex only** (simpler, misses the neutral reference)
- **Full optical flow pipeline** (richer signal, but more complex)

---

### Model architecture

```
onset_frame (224×224×3) ──┐
                           ├──► concatenate ──► (224×224×6) ──► patched input conv (6→96 channels)
apex_frame  (224×224×3) ──┘                                          │
                                                              Swin-Tiny Transformer
                                                            (pretrained on ImageNet,
                                                             first conv layer adapted)
                                                                      │
                                                           LayerNorm → Dropout → Linear
                                                                      │
                                                              emotion class logits
```

The patch embedding layer (first layer of Swin) is adapted from 3→6 input channels by **averaging the pretrained weights and doubling them** — this preserves the learned low-level feature detectors from ImageNet while accepting 6-channel input.

---

### Data flow (how this connects to your FTP loader)

```
CASMEDataLoader (FTP) 
      │  yields subject_path (e.g. ./data/casme/sub01_Part_A/)
      ▼
load_casme3_annotations()   ← reads the Excel file for onset/apex frame indices
      │
      ▼
CasmeBaselineDataset        ← finds clip folders, loads onset+apex, stacks to 6ch
      │
      ▼
DataLoader (batched)        ← feeds batches to the model
      │
      ▼
BaselineSwinMER             ← Swin-Tiny with 6-channel patch embedding
      │
      ▼
train() / evaluate()        ← LOSO cross-validation
```

---

### Requirements

```bash
pip install torch torchvision timm opencv-python-headless pandas openpyxl tqdm
```

### File layout (produced by CASMEDataLoader)
```
./data/casme/
    sub01_Part_A/              ← extracted from spNO.1.zip
        color/                 ← CASMEDataLoader grabs only color frames (no depth)
            EP01_01/           ← clip folder — matches 'Filename' in annotation
                00001.jpg
                00002.jpg
                ...
            EP01_02/
                ...
    sub02_Part_A/              ← extracted from spNO.2.zip
        color/
            ...
casme/CASME3_part_A_1.xls      ← annotation file
```

> **Subject ID mapping**: FTP zips are named `spNO.1.zip`, `spNO.2.zip`, etc.
> The annotation file uses plain integers (1, 2, 3...).
> `find_clip_folder()` handles this mapping by matching the numeric part.

---

### How to run

1. Make sure your `.env` file has `FTP_HOST`, `FTP_USER`, `FTP_PASS`
2. Run **Cell 1** — imports & config
3. Run **Cell 2** — annotation loader & path utilities
4. Run **Cell 3** — dataset class
5. Run **Cell 4** — load annotations, spot-check paths, and verify tensors
   - If you see `✗ NOT FOUND`, run the optional **tree diagnostic cell** to inspect the folder layout
6. Run **Cell 5** — build model and confirm 6-channel forward pass works
7. Run **Cell 6** — training functions (defines `train()` and `evaluate()`)
8. Run **Cell 7** — quick 3-epoch test on one subject before committing to full training
9. Run **Cell 8** — full LOSO cross-validation
10. Run **Cell 9** — inference on a single clip

---


## Cell 1 — Imports & Configuration
All dependencies and global settings in one place. Adjust `CONFIG` to change key hyperparameters without touching the rest of the code.

In [ ]:
%pip install timm

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import gc
import copy
import time
import glob
import warnings
warnings.filterwarnings('ignore')

# ── Numerical / Data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Image processing ──────────────────────────────────────────────────────────
import cv2

# ── Deep learning ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm                           # provides pretrained Swin Transformer

# ── Progress bars ─────────────────────────────────────────────────────────────
from tqdm.notebook import tqdm        # use tqdm.notebook for nicer Jupyter output

# ── Typing helpers ────────────────────────────────────────────────────────────
from typing import Optional, List, Dict, Tuple


# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL CONFIGURATION — change these values to experiment
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    # Paths
    "data_root":       "./casme_processed",   # root of the pre-processed dataset
    "sample_log_path": "sample_log.csv",      # where to save the list of sampled clips

    # Model
    "num_classes":     7,        # number of emotion categories
    "image_size":      224,      # Swin-Tiny expects 224×224
    "pretrained":      True,     # use ImageNet pretrained weights

    # Sampling — stratified: pick this many clips per emotion class
    # 100 × 7 classes = 700 clips (≈21% of the full 3,332-clip dataset)
    "sample_per_class": 100,

    # Training
    "epochs":          30,
    "batch_size":      16,
    "lr":              1e-4,     # head learning rate (backbone gets lr * 0.1)
    "weight_decay":    1e-4,
    "label_smoothing": 0.1,      # prevents overconfidence on small dataset
    "grad_clip":       1.0,      # prevents exploding gradients during fine-tuning
    "warmup_epochs":   5,        # LR linearly ramps up for first N epochs

    # Misc
    "seed":            42,
    "num_workers":     0,        # DataLoader worker processes
    "save_path":       "best_baseline.pth",
}

# ── Emotion label map ─────────────────────────────────────────────────────────
# Matches the 7 emotion categories present in casme_processed info.txt files.
LABEL_MAP = {
    "happiness":  0,
    "disgust":    1,
    "surprise":   2,
    "anger":      3,
    "fear":       4,
    "sadness":    5,
    "others":     6,
}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}   # reverse map for printing predictions

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

# ── Device selection ──────────────────────────────────────────────────────────
# Automatically picks GPU if available, otherwise CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"timm version: {timm.__version__}")
print(f"PyTorch version: {torch.__version__}")

## Cell 2 — Data Loader & Sampling Utilities
Functions for:
- Scanning `casme_processed/` and parsing every `info.txt` into a flat DataFrame
- Stratified random sampling (so we don't train on all 3,332 clips at once)
- Saving a sample log so every training run is reproducible and auditable
- Face detection & cropping (unchanged — applied to onset.jpg / apex.jpg)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# DATASET SCANNER
#
# casme_processed/ layout:
#   sub{ID}/
#       {video}_{onset}_{apex}/          ← clip folder
#           onset.jpg                    ← neutral face frame
#           apex.jpg                     ← peak expression frame
#           info.txt                     ← metadata (Subject, Video, Onset, Apex, Emotion)
#
# We walk the root, read every info.txt, and build a flat DataFrame.
# ─────────────────────────────────────────────────────────────────────────────

def _parse_info_txt(path: str) -> Optional[dict]:
    """
    Parse a single info.txt file into a dict with keys:
        subject, video, onset, apex, emotion

    Returns None if the file cannot be read or is malformed.
    """
    fields = {}
    try:
        with open(path, "r") as f:
            for line in f:
                line = line.strip()
                if ":" in line:
                    key, _, val = line.partition(":")
                    fields[key.strip().lower()] = val.strip()
    except OSError:
        return None

    required = {"subject", "video", "onset", "apex", "emotion"}
    if not required.issubset(fields):
        return None

    return {
        "subject":  int(fields["subject"]),
        "video":    fields["video"],
        "onset":    int(fields["onset"]),
        "apex":     int(fields["apex"]),
        "emotion":  fields["emotion"].lower(),
    }


def load_casme_processed(data_root: str) -> pd.DataFrame:
    """
    Walk `data_root` (the casme_processed folder) and collect metadata for
    every clip that has both onset.jpg and apex.jpg.

    Returns a DataFrame with columns:
        subject, clip_folder, onset_path, apex_path, emotion, emotion_id
    """
    records = []

    if not os.path.isdir(data_root):
        raise FileNotFoundError(f"data_root not found: {data_root}")

    for subj_dir in sorted(os.listdir(data_root)):
        subj_path = os.path.join(data_root, subj_dir)
        if not os.path.isdir(subj_path):
            continue

        for clip_dir in sorted(os.listdir(subj_path)):
            clip_path = os.path.join(subj_path, clip_dir)
            if not os.path.isdir(clip_path):
                continue

            info_path   = os.path.join(clip_path, "info.txt")
            onset_path  = os.path.join(clip_path, "onset.jpg")
            apex_path   = os.path.join(clip_path, "apex.jpg")

            # Skip clips where any required file is missing
            if not all(os.path.isfile(p) for p in [info_path, onset_path, apex_path]):
                continue

            info = _parse_info_txt(info_path)
            if info is None:
                continue

            emotion_id = LABEL_MAP.get(info["emotion"], LABEL_MAP.get("others", 6))

            records.append({
                "subject":     info["subject"],
                "clip_folder": clip_path,
                "onset_path":  onset_path,
                "apex_path":   apex_path,
                "emotion":     info["emotion"],
                "emotion_id":  emotion_id,
            })

    df = pd.DataFrame(records)
    print(f"Scanned {data_root!r}: found {len(df)} clips across {df['subject'].nunique()} subjects")
    print(f"Emotion distribution (full dataset):\n{df['emotion'].value_counts()}\n")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# STRATIFIED SAMPLING
#
# To keep training fast we pick a fixed number of clips per emotion class
# (controlled by CONFIG["sample_per_class"]).  This also gives a balanced
# training set, which matters on an imbalanced dataset like this one.
# ─────────────────────────────────────────────────────────────────────────────

def sample_clips(df: pd.DataFrame, n_per_class: int, seed: int = 42) -> pd.DataFrame:
    """
    Stratified random sample: pick at most `n_per_class` clips per emotion.

    For classes with fewer than `n_per_class` clips, all clips are kept
    (no upsampling).  Prints a per-class breakdown after sampling.
    """
    if df.empty or "emotion" not in df.columns:
        raise ValueError(
            "No clips found. Check that CONFIG['data_root'] points to the "
            "casme_processed/ folder and that it contains sub*/clip*/info.txt files."
        )

    parts = []
    for emotion_label, group in df.groupby("emotion"):
        n = min(n_per_class, len(group))
        parts.append(group.sample(n=n, random_state=seed))

    sampled = pd.concat(parts).reset_index(drop=True)
    print(f"Sampled {len(sampled)} clips ({n_per_class} per class, seed={seed}):")
    print(sampled["emotion"].value_counts().to_string())
    print()
    return sampled



# ─────────────────────────────────────────────────────────────────────────────
# SAMPLE LOG
#
# Writes the sampled DataFrame to a CSV so every training run can be
# reproduced exactly and audited (which clips were used, what emotion).
# ─────────────────────────────────────────────────────────────────────────────

def save_sample_log(df: pd.DataFrame, path: str) -> None:
    """Save the sampled clip list to `path` as a CSV."""
    log_cols = ["clip_folder", "subject", "emotion", "emotion_id"]
    df[log_cols].to_csv(path, index=False)
    print(f"Sample log saved → {path}  ({len(df)} rows)")


# ─────────────────────────────────────────────────────────────────────────────
# FACE DETECTION & CROPPING
# Isolating the face region removes irrelevant background information and
# ensures consistent spatial alignment across subjects.
# Using OpenCV's Haar cascade (no extra dependencies needed).
# ─────────────────────────────────────────────────────────────────────────────

# Load the face detector once (it's heavy to initialise repeatedly)
_face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def align_and_crop_face(img: np.ndarray, target_size: int = 224) -> np.ndarray:
    """
    Detect the face in an image, add a small margin, and resize to target_size.

    If no face is detected (which can happen with extreme head poses or
    poor lighting), falls back to a centre crop of the whole image.
    This graceful fallback prevents the pipeline from crashing on hard samples.
    """
    gray  = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    faces = _face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60)
    )

    if len(faces) > 0:
        x, y, w, h = faces[0]
        margin = int(0.15 * min(w, h))
        x1 = max(0, x - margin)
        y1 = max(0, y - margin)
        x2 = min(img.shape[1], x + w + margin)
        y2 = min(img.shape[0], y + h + margin)
        crop = img[y1:y2, x1:x2]
    else:
        # Fallback: square centre crop
        h, w  = img.shape[:2]
        s     = min(h, w)
        y1    = (h - s) // 2
        x1    = (w - s) // 2
        crop  = img[y1:y1+s, x1:x1+s]

    return cv2.resize(crop, (target_size, target_size))


print("Cell 2 loaded ✓")

## Cell 3 — Dataset Class
The `CasmeBaselineDataset` handles loading and preparing individual training samples.  
Each sample is a **6-channel tensor** = onset frame (RGB) stacked with apex frame (RGB).

Because `casme_processed/` already has `onset.jpg` and `apex.jpg` extracted for every clip,
the dataset can load them directly by path — no frame-index arithmetic needed.

Blank tensors are still returned for any file that fails to load, so training
never crashes on a corrupt or missing image.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IMAGE TRANSFORMS
# Standard ImageNet normalisation values — used because our Swin backbone
# was pretrained on ImageNet with these statistics.
# We apply them to EACH 3-channel frame before concatenating.
# ─────────────────────────────────────────────────────────────────────────────

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Transform applied to each individual frame (onset or apex) before stacking
_frame_transform_base = transforms.Compose([
    transforms.ToTensor(),                                # numpy HWC → torch CHW, scale to [0,1]
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),   # normalise per ImageNet stats
])

_frame_transform_aug = transforms.Compose([
    transforms.ToTensor(),
    # Horizontal flip: micro-expressions are roughly symmetric so this is safe
    transforms.RandomHorizontalFlip(p=0.5),
    # Small brightness/contrast jitter to improve robustness to lighting
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


# ─────────────────────────────────────────────────────────────────────────────
# DATASET CLASS
# ─────────────────────────────────────────────────────────────────────────────

class CasmeBaselineDataset(Dataset):
    """
    PyTorch Dataset for the casme_processed baseline model.

    Each __getitem__ call:
      1. Reads onset_path and apex_path directly from the DataFrame row
         (these are absolute paths to pre-extracted .jpg files)
      2. Crops both frames to the face region
      3. Applies the same augmentation to BOTH frames (important: the same
         random flip/jitter must be applied to both so they stay aligned)
      4. Stacks them channel-wise: [onset_R, onset_G, onset_B, apex_R, apex_G, apex_B]
      5. Returns (6-channel tensor, emotion class index)
    """

    def __init__(
        self,
        annotations: pd.DataFrame,
        image_size:  int  = 224,
        augment:     bool = False,
    ):
        self.df        = annotations.reset_index(drop=True)
        self.size      = image_size
        self.augment   = augment
        self.transform = _frame_transform_aug if augment else _frame_transform_base

        # Pre-build a blank frame (used as fallback when an image fails to load)
        self._blank = np.zeros((image_size, image_size, 3), dtype=np.uint8)

    def __len__(self):
        return len(self.df)

    def _load_jpg(self, path: str) -> np.ndarray:
        """
        Load a JPEG image from `path`, convert BGR→RGB, then face-crop.
        Returns a blank image if loading fails.
        """
        img = cv2.imread(path)
        if img is None:
            return self._blank.copy()
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return align_and_crop_face(img_rgb, self.size)

    def __getitem__(self, idx: int):
        row   = self.df.iloc[idx]
        label = int(row["emotion_id"])

        # ── Load onset and apex frames from pre-known paths ───────────────────
        onset_img = self._load_jpg(row["onset_path"])
        apex_img  = self._load_jpg(row["apex_path"])

        # ── Apply transforms ──────────────────────────────────────────────────
        # For augmentation we want the SAME random transformation applied to
        # both frames (e.g. both flipped, or neither) so they stay aligned.
        if self.augment:
            seed = torch.randint(0, 2**32, (1,)).item()
            torch.manual_seed(seed);  onset_tensor = self.transform(onset_img)
            torch.manual_seed(seed);  apex_tensor  = self.transform(apex_img)
        else:
            onset_tensor = self.transform(onset_img)
            apex_tensor  = self.transform(apex_img)

        # ── Stack into a 6-channel tensor ─────────────────────────────────────
        # Shape: (6, 224, 224) — channels 0-2 are onset, channels 3-5 are apex
        six_channel = torch.cat([onset_tensor, apex_tensor], dim=0)

        return six_channel, label


# ─────────────────────────────────────────────────────────────────────────────
# DATASET INSPECTION HELPER
# ─────────────────────────────────────────────────────────────────────────────

def inspect_dataset_sample(dataset: CasmeBaselineDataset, n: int = 3):
    """Load a few samples and print their shapes and labels to confirm the pipeline works."""
    print(f"Dataset size: {len(dataset)} samples")
    print(f"Inspecting {n} samples...\n")
    for i in range(min(n, len(dataset))):
        tensor, label = dataset[i]
        print(f"  Sample {i}: tensor shape={tuple(tensor.shape)}, "
              f"label={label} ({ID_TO_LABEL.get(label, '?')})")
        print(f"           onset channels min={tensor[:3].min():.2f} max={tensor[:3].max():.2f}")
        print(f"           apex  channels min={tensor[3:].min():.2f} max={tensor[3:].max():.2f}")

print("Cell 3 loaded ✓")

## Cell 4 — Load Dataset, Sample & Verify

Three steps:
1. **Scan** `casme_processed/` → flat DataFrame of all 3,332 clips
2. **Sample** — stratified random subset (`sample_per_class` clips per emotion class)
3. **Save sample log** → `sample_log.csv` (auditable record of which clips were used)
4. **Inspect** a few sample tensors to confirm the full pipeline works

In [ ]:
# ── Step 1: Scan casme_processed/ ─────────────────────────────────────────────
# Reads every info.txt → one row per clip with onset_path, apex_path, emotion
all_clips = load_casme_processed(CONFIG["data_root"])

# ── Step 2: Stratified random sample ──────────────────────────────────────────
# Picks CONFIG["sample_per_class"] clips per emotion class.
# 100 × 7 classes = 700 clips.  Change sample_per_class in CONFIG to adjust.
annotations = sample_clips(all_clips, CONFIG["sample_per_class"], seed=CONFIG["seed"])

# ── Step 3: Save sample log ────────────────────────────────────────────────────
# Written to sample_log.csv — lists every clip_folder + emotion used this run.
save_sample_log(annotations, CONFIG["sample_log_path"])

# Quick preview
annotations.head(5)

In [ ]:
# ── Step 4: Build test dataset and verify tensors ─────────────────────────────
# Confirms the full pipeline (load → face-crop → 6ch stack → tensor) works
# before we start any training.
test_ds = CasmeBaselineDataset(
    annotations,
    image_size=CONFIG["image_size"],
    augment=False,
)
inspect_dataset_sample(test_ds, n=3)

## Cell 5 — Model Definition
The `BaselineSwinMER` model adapts a **pretrained Swin-Tiny Transformer** to accept 6-channel input.

The key trick is adapting the **patch embedding layer** (the very first layer, which converts image patches to tokens) from 3→6 input channels. We do this by copying the pretrained 3-channel weights and averaging them — this way we don't lose the pretrained knowledge.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LABEL SMOOTHING LOSS
# On small datasets, hard (one-hot) labels cause the model to become
# overconfident and overfit. Label smoothing distributes a small probability
# mass to all other classes, acting as a regulariser.
# ─────────────────────────────────────────────────────────────────────────────

class LabelSmoothingCrossEntropy(nn.Module):
    """Cross-entropy loss with label smoothing."""

    def __init__(self, num_classes: int, smoothing: float = 0.1):
        super().__init__()
        self.smoothing = smoothing
        self.n_classes  = num_classes

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # confidence = probability assigned to the correct class
        confidence = 1.0 - self.smoothing
        # smooth_val = probability distributed to all OTHER classes
        smooth_val = self.smoothing / (self.n_classes - 1)

        # Build soft target distribution
        soft_targets = torch.full_like(logits, smooth_val)
        soft_targets.scatter_(1, targets.unsqueeze(1), confidence)

        # Compute cross-entropy against soft targets
        log_probs = F.log_softmax(logits, dim=1)
        return -(soft_targets * log_probs).sum(dim=1).mean()


# ─────────────────────────────────────────────────────────────────────────────
# BASELINE MODEL: 6-CHANNEL SWIN TRANSFORMER
# ─────────────────────────────────────────────────────────────────────────────

class BaselineSwinMER(nn.Module):
    """
    Swin-Tiny Transformer adapted for 6-channel input (onset + apex frames).

    Architecture:
        Input: (B, 6, 224, 224)
          → Adapted patch embedding (6ch → 96 tokens)
          → Swin Transformer stages (pretrained weights preserved)
          → Global average pool → LayerNorm
          → Dropout → Linear classifier
        Output: (B, num_classes) logits

    The first-layer weight adaptation:
        Pretrained weight shape: (96, 3, 4, 4)  [out_ch, in_ch, kH, kW]
        We need shape:           (96, 6, 4, 4)
        Strategy: tile the 3-channel weights twice and scale by 0.5
        This preserves the learned low-level feature detectors (edges,
        textures) from ImageNet while accepting the extra channels.
    """

    def __init__(self, num_classes: int = 7, pretrained: bool = True):
        super().__init__()

        # ── Load Swin-Tiny with original 3-channel head removed ───────────────
        # num_classes=0 tells timm to return the feature vector, not logits
        self.backbone = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=pretrained,
            num_classes=0,      # remove original classification head
        )

        # ── Adapt the patch embedding from 3 → 6 input channels ──────────────
        # The patch embedding is a Conv2d that maps (3, 224, 224) → (96, 56, 56)
        # We need it to map (6, 224, 224) → (96, 56, 56)
        self._adapt_patch_embedding()

        # ── New classification head ───────────────────────────────────────────
        # Feature dimension of Swin-Tiny is 768
        feat_dim = self.backbone.num_features   # 768
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim),   # stabilise feature magnitudes
            nn.Dropout(p=0.3),        # regularisation — important on small datasets
            nn.Linear(feat_dim, num_classes),
        )

    def _adapt_patch_embedding(self):
        """
        Modify the first Conv2d of the Swin patch embedding to accept 6 channels.

        Original: weight shape (96, 3, 4, 4)
        New:      weight shape (96, 6, 4, 4)

        We initialise the new 6-channel weight by tiling the 3-channel pretrained
        weight and scaling by 0.5 to maintain the same expected output magnitude.
        """
        # Access the Conv2d layer inside the patch embedding module
        old_conv = self.backbone.patch_embed.proj

        # Get the pretrained weights: shape (96, 3, 4, 4)
        old_weight = old_conv.weight.data
        old_bias   = old_conv.bias.data if old_conv.bias is not None else None

        # Create new Conv2d with 6 input channels, same other settings
        new_conv = nn.Conv2d(
            in_channels=6,                  # ← changed from 3
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=(old_conv.bias is not None),
        )

        # Initialise: tile pretrained weights across the 6 channels
        # torch.cat([w, w], dim=1) gives shape (96, 6, 4, 4)
        # Scale by 0.5 so summed activations have the same expected magnitude
        new_conv.weight.data = torch.cat([old_weight, old_weight], dim=1) * 0.5
        if old_bias is not None:
            new_conv.bias.data = old_bias.clone()

        # Replace the old Conv2d with our new one
        self.backbone.patch_embed.proj = new_conv
        print(f"Patch embedding adapted: 3ch → 6ch (weight shape: {new_conv.weight.shape})")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (B, 6, 224, 224)  — concatenated onset + apex frames
        returns: (B, num_classes) — raw logits (apply softmax for probabilities)
        """
        features = self.backbone(x)       # (B, 768) after global avg pool
        return self.classifier(features)  # (B, num_classes)


# ─────────────────────────────────────────────────────────────────────────────
# PARAMETER COUNT & MODEL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

def count_parameters(model: nn.Module) -> Dict[str, int]:
    """Count trainable and frozen parameters separately."""
    trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen      = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return {"trainable": trainable, "frozen": frozen, "total": trainable + frozen}


# Build model and print summary
model = BaselineSwinMER(
    num_classes=CONFIG["num_classes"],
    pretrained=CONFIG["pretrained"]
).to(DEVICE)

param_counts = count_parameters(model)
print(f"\nModel parameters:")
print(f"  Trainable:  {param_counts['trainable']:,}")
print(f"  Total:      {param_counts['total']:,}")

# Confirm the model accepts 6-channel input
dummy = torch.zeros(2, 6, 224, 224).to(DEVICE)   # batch of 2 fake samples
with torch.no_grad():
    out = model(dummy)
print(f"\nForward pass OK: input {tuple(dummy.shape)} → output {tuple(out.shape)}")

## Cell 6 — Training & Evaluation Functions
The training loop with:
- **Layer-wise learning rate decay** — backbone (pretrained) trains 10× slower than the new head
- **Cosine annealing with linear warmup** — smooth LR schedule that works well for transformer fine-tuning
- **Gradient clipping** — prevents exploding gradients when fine-tuning on a small dataset
- **Best model checkpointing** — saves only when validation accuracy improves

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SINGLE EPOCH: TRAINING PASS
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(
    model:     nn.Module,
    loader:    DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device:    torch.device,
    grad_clip: float,
) -> Tuple[float, float]:
    """Run one full pass over the training set. Returns (avg_loss, accuracy)."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch_tensors, labels in tqdm(loader, desc="  train", leave=False):
        batch_tensors = batch_tensors.to(device)   # (B, 6, 224, 224)
        labels        = labels.to(device)           # (B,)

        # Forward pass
        logits = model(batch_tensors)               # (B, num_classes)
        loss   = criterion(logits, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()

        # Clip gradients — prevents a large gradient from a single hard sample
        # from causing a destabilising weight update
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        optimizer.step()

        # Accumulate metrics
        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)


# ─────────────────────────────────────────────────────────────────────────────
# SINGLE EPOCH: EVALUATION PASS
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()   # disables gradient tracking — faster and uses less memory
def evaluate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
    device:    torch.device,
) -> Tuple[float, float]:
    """Evaluate model on validation set. Returns (avg_loss, accuracy)."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for batch_tensors, labels in tqdm(loader, desc="  eval ", leave=False):
        batch_tensors = batch_tensors.to(device)
        labels        = labels.to(device)

        logits = model(batch_tensors)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)


# ─────────────────────────────────────────────────────────────────────────────
# FULL TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────

def train(
    model:        nn.Module,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    device:       torch.device,
    config:       dict,
    save_path:    str = "best_baseline.pth",
) -> List[Dict]:
    """
    Full training loop with:
      - Layer-wise LR: backbone trains at lr*0.1, head at lr
        (protects pretrained features from being overwritten too quickly)
      - Warmup + cosine LR schedule
      - Label smoothing loss
      - Best checkpoint saving

    Returns training history (list of per-epoch metric dicts).
    """
    epochs    = config["epochs"]
    lr        = config["lr"]
    wd        = config["weight_decay"]
    grad_clip = config["grad_clip"]
    warmup    = config["warmup_epochs"]

    criterion = LabelSmoothingCrossEntropy(
        num_classes=config["num_classes"],
        smoothing=config["label_smoothing"]
    )

    # ── Layer-wise learning rates ────────────────────────────────────────────
    # The pretrained backbone gets a 10× lower LR than the new head.
    # This way we slowly fine-tune the pretrained representations rather
    # than overwriting them with random gradients from the new head.
    backbone_params = list(model.backbone.parameters())
    head_params     = list(model.classifier.parameters())

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": lr * 0.1},   # slow: preserve pretrained features
        {"params": head_params,     "lr": lr},          # fast: learn task-specific mapping
    ], weight_decay=wd)

    # ── Learning rate schedule ────────────────────────────────────────────────
    # Phase 1 (warmup): LR linearly increases from 0 to full value
    # Phase 2 (cosine): LR smoothly decays following a cosine curve
    # This avoids large early updates that could damage pretrained weights.
    def lr_lambda(epoch: int) -> float:
        if epoch < warmup:
            return (epoch + 1) / warmup   # linear warmup
        progress = (epoch - warmup) / max(1, epochs - warmup)
        return 0.5 * (1.0 + np.cos(np.pi * progress))   # cosine decay

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # ── Training loop ─────────────────────────────────────────────────────────
    best_val_acc = 0.0
    history      = []

    for epoch in range(1, epochs + 1):
        current_lr = optimizer.param_groups[1]["lr"]   # head LR (the faster one)

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device, grad_clip
        )
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        # Record metrics for this epoch
        entry = {
            "epoch":      epoch,
            "lr":         current_lr,
            "train_loss": train_loss,
            "train_acc":  train_acc,
            "val_loss":   val_loss,
            "val_acc":    val_acc,
        }
        history.append(entry)

        print(
            f"Epoch {epoch:03d}/{epochs} | lr={current_lr:.2e} | "
            f"train loss={train_loss:.4f} acc={train_acc:.3f} | "
            f"val loss={val_loss:.4f} acc={val_acc:.3f}"
            + (" ← best" if val_acc > best_val_acc else "")
        )

        # Save checkpoint whenever validation accuracy improves
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "epoch":     epoch,
                "model":     model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "val_acc":   val_acc,
                "config":    config,
            }, save_path)

    print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")
    print(f"Best checkpoint saved to: {save_path}")
    return history


# ─────────────────────────────────────────────────────────────────────────────
# HISTORY PLOTTING (optional, requires matplotlib)
# ─────────────────────────────────────────────────────────────────────────────

def plot_training_history(history: List[Dict]):
    """Plot loss and accuracy curves from training history."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib not installed — skipping plot")
        return

    epochs     = [h["epoch"]      for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss   = [h["val_loss"]   for h in history]
    train_acc  = [h["train_acc"]  for h in history]
    val_acc    = [h["val_acc"]    for h in history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, train_loss, label="Train loss")
    ax1.plot(epochs, val_loss,   label="Val loss")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title("Loss curves"); ax1.legend()

    ax2.plot(epochs, train_acc, label="Train acc")
    ax2.plot(epochs, val_acc,   label="Val acc")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy curves"); ax2.legend()

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150)
    plt.show()
    print("Plot saved to training_history.png")

print("Cell 6 loaded ✓")

## Cell 7 — Quick Single-Subject Test
Before running full LOSO (which can take hours), run a **quick sanity check** on just one held-out subject to confirm the whole pipeline works end-to-end.

In [ ]:
# ── Pick one subject as a quick test ─────────────────────────────────────────
all_subjects = annotations["subject"].unique()
test_subject = all_subjects[0]   # hold out the first subject
print(f"Quick test: training on all except subject '{test_subject}', "
      f"validating on subject '{test_subject}'")

# ── Split ─────────────────────────────────────────────────────────────────────
quick_train_df = annotations[annotations["subject"] != test_subject]
quick_val_df   = annotations[annotations["subject"] == test_subject]
print(f"Train: {len(quick_train_df)} clips | Val: {len(quick_val_df)} clips")

# ── Datasets & loaders ────────────────────────────────────────────────────────
quick_train_ds = CasmeBaselineDataset(
    quick_train_df, image_size=CONFIG["image_size"], augment=True
)
quick_val_ds = CasmeBaselineDataset(
    quick_val_df, image_size=CONFIG["image_size"], augment=False
)

quick_train_loader = DataLoader(
    quick_train_ds, batch_size=CONFIG["batch_size"],
    shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True
)
quick_val_loader = DataLoader(
    quick_val_ds, batch_size=CONFIG["batch_size"],
    shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True
)

# ── Run a short 3-epoch test ──────────────────────────────────────────────────
# We override epochs=3 just to confirm nothing crashes before committing
# to a full LOSO run.
quick_config = {**CONFIG, "epochs": 3, "save_path": "quick_test.pth"}

quick_model = BaselineSwinMER(
    num_classes=CONFIG["num_classes"],
    pretrained=CONFIG["pretrained"]
).to(DEVICE)

quick_history = train(
    quick_model, quick_train_loader, quick_val_loader,
    DEVICE, quick_config, save_path=quick_config["save_path"]
)
print("\nQuick test passed ✓ — safe to run full LOSO in Cell 8")

## Cell 8 — Full LOSO Cross-Validation
**Leave-One-Subject-Out (LOSO)** is the standard evaluation protocol for CASME datasets.

For each subject:
1. Train on all *other* subjects
2. Evaluate on that subject
3. Report mean ± std accuracy across all subjects

This prevents data leakage (the same person appearing in both train and test), which would artificially inflate accuracy.

**Integration with CASMEDataLoader**: if you're streaming data subject-by-subject from the FTP loader, call `loader.get_next_subject()` to get each subject path, then update `CONFIG["data_root"]` accordingly, or simply ensure all subjects are downloaded before running LOSO.

In [ ]:
def run_loso(
    annotations: pd.DataFrame,
    config:      dict,
    device:      torch.device,
) -> pd.DataFrame:
    """
    Leave-One-Subject-Out cross-validation.

    For each subject:
      - Trains a fresh model on all other subjects
      - Evaluates on the held-out subject
      - Saves the best checkpoint per subject

    Returns a DataFrame with per-subject results.
    """
    subjects = sorted(annotations["subject"].unique())
    results  = []

    print(f"Starting LOSO with {len(subjects)} subjects...")
    print("=" * 60)

    for i, held_out_subject in enumerate(subjects):
        print(f"\n[{i+1}/{len(subjects)}] Held-out subject: {held_out_subject}")
        print("-" * 40)

        # ── Split: this subject is validation, all others are training ────────
        train_df = annotations[annotations["subject"] != held_out_subject]
        val_df   = annotations[annotations["subject"] == held_out_subject]
        print(f"  Train: {len(train_df)} clips | Val: {len(val_df)} clips")

        # ── Build datasets ────────────────────────────────────────────────────
        train_ds = CasmeBaselineDataset(
            train_df, image_size=config["image_size"], augment=True
        )
        val_ds = CasmeBaselineDataset(
            val_df, image_size=config["image_size"], augment=False
        )

        train_loader = DataLoader(
            train_ds, batch_size=config["batch_size"],
            shuffle=True, num_workers=config["num_workers"], pin_memory=True
        )
        val_loader = DataLoader(
            val_ds, batch_size=config["batch_size"],
            shuffle=False, num_workers=config["num_workers"], pin_memory=True
        )

        # ── Fresh model for each fold ─────────────────────────────────────────
        # IMPORTANT: we must create a new model each iteration, not reuse the
        # previous one, to avoid information leaking between folds.
        fold_model = BaselineSwinMER(
            num_classes=config["num_classes"],
            pretrained=config["pretrained"]
        ).to(device)

        save_path = config["save_path"].replace(".pth", f"_subj{held_out_subject}.pth")

        # ── Train ─────────────────────────────────────────────────────────────
        history = train(
            fold_model, train_loader, val_loader,
            device, config, save_path=save_path
        )

        # Record the best validation accuracy achieved in this fold
        best_acc = max(h["val_acc"] for h in history)
        best_epoch = max(history, key=lambda h: h["val_acc"])["epoch"]

        results.append({
            "subject":    held_out_subject,
            "n_val":      len(val_df),
            "best_acc":   best_acc,
            "best_epoch": best_epoch,
            "checkpoint": save_path,
        })

        print(f"  Subject {held_out_subject}: best val_acc = {best_acc:.4f} (epoch {best_epoch})")

        # ── Free GPU memory between folds ─────────────────────────────────────
        del fold_model, train_loader, val_loader, train_ds, val_ds
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # ── Summary ───────────────────────────────────────────────────────────────
    results_df  = pd.DataFrame(results)
    mean_acc    = results_df["best_acc"].mean()
    std_acc     = results_df["best_acc"].std()



    print("\n" + "==" * 60)
    print("LOSO RESULTS SUMMARY")
    print("=" * 60)
    print(results_df.to_string(index=False))
    print(f"\nMean accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
    print(f"Best subject:  {results_df.loc[results_df['best_acc'].idxmax(), 'subject']} "
          f"({results_df['best_acc'].max():.4f})")
    print(f"Worst subject: {results_df.loc[results_df['best_acc'].idxmin(), 'subject']} "
          f"({results_df['best_acc'].min():.4f})")

    results_df.to_csv("loso_results.csv", index=False)
    print("\nResults saved to loso_results.csv")

    return results_df


# ── Run LOSO ─────────────────────────────────────────────────────────────────
# Comment this out and run Cell 7 first to confirm the pipeline works.
loso_results = run_loso(annotations, CONFIG, DEVICE)

## Cell 9 — Inference on a Single Clip
Once training is done, use this to predict the emotion for any clip you have locally — useful for debugging or demonstrating the model.

In [ ]:
@torch.no_grad()
def predict_clip(
    model:      nn.Module,
    onset_path: str,
    apex_path:  str,
    device:     torch.device,
    image_size: int = 224,
) -> Tuple[str, float, Dict[str, float]]:
    """
    Predict the emotion for a single clip given its onset and apex image paths.

    Args:
        model:       trained BaselineSwinMER
        onset_path:  path to the onset frame (onset.jpg)
        apex_path:   path to the apex  frame (apex.jpg)
        device:      torch device

    Returns:
        (predicted_label, confidence, all_class_probs)
    """
    model.eval()

    blank = np.zeros((image_size, image_size, 3), dtype=np.uint8)

    def _load(path):
        img = cv2.imread(path)
        if img is None:
            return blank
        return align_and_crop_face(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), image_size)

    onset_img = _load(onset_path)
    apex_img  = _load(apex_path)

    onset_t = _frame_transform_base(onset_img)
    apex_t  = _frame_transform_base(apex_img)
    six_ch  = torch.cat([onset_t, apex_t], dim=0).unsqueeze(0).to(device)  # (1, 6, 224, 224)

    logits = model(six_ch)
    probs  = F.softmax(logits, dim=1)[0]

    pred_id    = probs.argmax().item()
    confidence = probs[pred_id].item()
    all_probs  = {ID_TO_LABEL.get(i, str(i)): probs[i].item() for i in range(len(probs))}

    return ID_TO_LABEL.get(pred_id, str(pred_id)), confidence, all_probs


# ── Example usage ─────────────────────────────────────────────────────────────
# Uses the first row of the sampled annotations — replace with any row you like.
example_row = annotations.iloc[0]

# Load best model weights (requires Cell 7 or 8 to have completed successfully)
inference_model = BaselineSwinMER(
    num_classes=CONFIG["num_classes"], pretrained=False
).to(DEVICE)

checkpoint = torch.load(CONFIG["save_path"], map_location=DEVICE)
inference_model.load_state_dict(checkpoint["model"])

pred_label, confidence, all_probs = predict_clip(
    inference_model,
    onset_path=example_row["onset_path"],
    apex_path=example_row["apex_path"],
    device=DEVICE,
)

print(f"Clip:         {example_row['clip_folder']}")
print(f"Subject:      {example_row['subject']}")
print(f"Ground truth: {example_row['emotion']}")
print(f"Predicted:    {pred_label} (confidence: {confidence:.1%})")
print(f"\nAll class probabilities:")
for emotion, prob in sorted(all_probs.items(), key=lambda x: -x[1]):
    bar = "█" * int(prob * 30)
    print(f"  {emotion:12s} {prob:.3f}  {bar}")